**COLLECTING DATA**
_________________________________________________________________________

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ValueError: mount failed

**LOAD LIBRARY**
___________________________________________________________________________

In [ ]:
# Import Library
import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy import stats
from datetime import datetime, timedelta
import matplotlib.dates as mdates
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import norm
from scipy.stats import kstest, norm
from scipy.stats import zscore

# Matikan Warning (Opsional)
warnings.filterwarnings("ignore")

# Sklearn untuk preprocessing dan evaluasi model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

# TensorFlow dan Keras untuk membangun model LSTM
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import Huber

**EXPLORATORY DATA ANALYSIS**
__________________________________________________________________________

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/indofood.csv")
data

In [ ]:
# Melihat Informasi Data
data.info()

In [ ]:
# Melihat Statistik Deskriptif Data
data.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(data["Close"], color="blue", label="Close Price")
plt.title("Stock Price Movement of PT Indofood Sukses Makmur Tbk")
plt.xlabel("Date")
plt.ylabel("Stock Price (IDR)")
plt.legend(loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
data_values = data["Close"]
decomposition = seasonal_decompose(data_values, model="additive", period=7)

# Ambil komponen-komponennya
trend = decomposition.trend
seasonal = decomposition.seasonal
residual = decomposition.resid

# Plot hasil dekomposisi
plt.figure(figsize=(12, 8))

plt.subplot(411)
plt.plot(data_values, label="Original", color="orange")
plt.legend(loc="upper left")

plt.subplot(412)
plt.plot(trend, label="Trend", color="blue")
plt.legend(loc="upper left")

plt.subplot(413)
plt.plot(seasonal, label="Seasonal", color="green")
plt.legend(loc="upper left")

plt.subplot(414)
plt.plot(residual, label="Residual", color="red")
plt.legend(loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# Melihat QQ Plot Distribusi Harga Saham
stats.probplot(data["Close"], dist="norm", plot=plt)
plt.title("QQ Plot of Stock Price")
plt.show()

**DATA PREPROCESSING**
_____________________________________________

> Handle Missing Values

In [ ]:
# Penanganan Missing Values
missing_values = data.isnull().sum()
missing_values

> Handle Duplicate Data

In [ ]:
# Penanganan Data Duplicate
duplicates = data.duplicated(keep=False)
print("Jumlah data duplikat:", duplicates.sum())

> Ubah Tipe Data

In [ ]:
# Mengubah Tipe Data
data["Date"] = pd.to_datetime(data["Date"])
data.info()

> Handle Outliers



In [ ]:
# Hitung z-score untuk setiap kolom numerik
z_scores = np.abs(zscore(data.select_dtypes(include=[np.number])))

# Tetapkan threshold z-score (misalnya 3.0)
threshold = 3.0

# Identifikasi baris yang memiliki outlier
outliers = (z_scores > threshold).any(axis=1)

# Cetak jumlah data outlier
print("Jumlah data outlier:", outliers.sum())

# Hitung mean dari data, tetapi hanya untuk kolom numerik
mean_value = data.select_dtypes(include=np.number).mean()

# Menangani outlier: Ganti outlier dengan mean kolom masing-masing
for column in data.select_dtypes(include=np.number).columns:
    data[column] = np.where(outliers, mean_value[column], data[column])

# Buat DataFrame
df_data = pd.DataFrame(data)

# Menampilkan hasil
print("Data Setelah Penanganan Outlier (Diganti dengan Mean):")
df_data

> Features Selection

In [ ]:
data = data.drop(columns=["Open", "High", "Low", "Volume"])
data

> Normalization : Log Transform

In [ ]:
data["Normalized"] = np.log(data["Close"])
data

**FUZZY TIME SERIES - LONG SHORT TERM MEMORY MODEL CONSTRUCTION**
___________________________________________

> Hitung Parameter Intercept & Slope

In [ ]:
y = data["Normalized"].values
t = np.arange(1, len(y) + 1)  # t_i: waktu pengamatan ke-i

# Hitung rata-rata t dan y
t_mean = np.mean(t)
y_mean = np.mean(y)

# Hitung slope (β₁)
numerator = np.sum((t - t_mean) * (y - y_mean))
denominator = np.sum((t - t_mean) ** 2)
slope = numerator / denominator

# Hitung intercept (β₀)
intercept = y_mean - slope * t_mean

# Cetak hasil
print("Slope (β₁):", slope)
print("Intercept (β₀):", intercept)



> Menghitung Trend Value & Relative Trend Value



In [ ]:
Ut = intercept + slope * t
Vt = y / Ut * 100

df_utvt = pd.DataFrame({"Waktu": t, "Trend Value": Ut, "Relative Trend Value": Vt})
df_utvt

> Membagi Data Menjadi 2 Sub Group Berdasarkan Median

In [ ]:
medianv = np.median(Vt)
P = Vt[Vt <= medianv]
Q = Vt[Vt > medianv]

In [ ]:
P

In [ ]:
Q

> Menghitung Perbedaan Absolut Setiap Sub Bab Beserta Rata - Ratanya

In [ ]:
# Hitung perbedaan absolut (Pd & Qd)
Pd = np.abs(np.diff(P))
Qd= np.abs(np.diff(Q))

df_PdQd = pd.DataFrame({"Perbedaan Pd": Pd, "Perbedaan Qd": Qd})
df_PdQd

In [ ]:
# Rata - Rata Perbedaan Pertama (Pv & Qv)
Pv = Pd.mean()
Qv = Qd.mean()

print("Rata - Rata Perbedaan Pertama (Pv):", Pv)
print("Rata - Rata Perbedaan Pertama (Qv):", Qv)

> Menentukan Faktor Pengambilan Keputusan (ϑ)

In [ ]:
# Menghitung e
e = Pv / Qv

# Menghitung floor(e)
e_floor = math.floor(e)

# Menghitung ϑ
ϑ = 10 ** e_floor * (Pv / Qv)
print("Faktor Pengambilan Keputusan :", ϑ)

> Mendapatkan Number of Interval (k)

In [ ]:
# Menghitung min dan max dari V
min_V = np.min(data["Normalized"])
max_V = np.max(data["Normalized"])
l_t = 1.0

# Menentukan n dengan menggunakan Sturges' Rule
N = len(data["Normalized"])
n = int(1 + np.log2(N))

# Menghitung jumlah interval output fuzzy k
k = ((min_V - ϑ) / (max_V + ϑ)) + (l_t * n)

print("Jumlah interval output fuzzy k:", k)

> Mendapatkan Nilai Universe of Discourse (UOD)

In [ ]:
# Melihat Statistik Deskriptif Kolom Close
data_min = np.min(data["Normalized"])
data_max = np.max(data["Normalized"])
delta = np.std(data["Normalized"])

# Menghitung UOD
UOD = (data_min - delta, data_max + delta)

print(f"Minimum value (min(y)): {data_min}")
print(f"Maximum value (max(y)): {data_max}")
print(f"Standard deviation (δ): {delta}")
print(f"Universe of Discourse (UOD): {UOD}")

> Membagi Nilai UOD Sejumlah k

In [ ]:
data = data["Normalized"]

# Buffer untuk memperluas UoD (misalnya pakai theta)
U_min = np.min(data) - ϑ
U_max = np.max(data) + ϑ

# Jumlah interval yang ingin dibagi
k = 12
w = k  # karena w = jumlah interval

# Membagi UoD menjadi k interval dengan panjang sama
boundaries = np.linspace(U_min, U_max, k + 1)

# Membentuk list interval dan titik tengahnya
intervals = []
midpoints = []

boundaries = np.linspace(U_min, U_max, k + 1)
midpoints = [(boundaries[i] + boundaries[i + 1]) / 2 for i in range(k)]

for i in range(k):
    lower = boundaries[i]
    upper = boundaries[i + 1]
    intervals.append((lower, upper))
    mid = (lower + upper) / 2
    midpoints.append(mid)

# Tampilkan hasil
print("Interval UoD:")
for i, (low, high) in enumerate(intervals):
    print(f"Interval {i+1}: [{low:.2f}, {high:.2f}] | Midpoint: {midpoints[i]:.2f}")

> Fuzzifikasi Data

In [ ]:
# Fungsi fuzzify untuk interval berbasis array
def fuzzify(value, interval_array):
    # Memeriksa apakah nilai adalah nilai tunggal (skalar)
    if np.isscalar(value):
        for i in range(len(interval_array)):
            # Access elements of the tuple using [0] and [1]
            lower_bound = interval_array[i][0]
            upper_bound = interval_array[i][1]
            if lower_bound <= value <= upper_bound:
                return i + 1  # Mengembalikan indeks interval (dimulai dari 1)
        return None  # Mengembalikan None jika tidak ada interval yang cocok
    # Jika nilai adalah array, memproses setiap elemen secara terpisah
    else:
        return [fuzzify(v, interval_array) for v in value]

# Mengubah daftar 'intervals' menjadi array NumPy
interval_array = np.array(intervals)

# Create a DataFrame to store results
fuzzified_data = pd.DataFrame({'Normalized': data}) # Create a DataFrame with the 'Normalized' column

# Menerapkan fuzzifikasi pada kolom 'Normalized'
fuzzified_data["Fuzzified"] = fuzzified_data["Normalized"].apply(lambda x: fuzzify(x, interval_array))
df_fuzzified = pd.DataFrame(fuzzified_data)

# Menampilkan hasil
print("\nData yang telah difuzzifikasi")
print(fuzzified_data)

> Membentuk Fuzzy Logical Relationship

In [ ]:
series = fuzzified_data["Fuzzified"]

acf_values = acf(series, nlags=10, fft=False)  # Batasi hanya 10 lag

threshold = 0.5  # Threshold yang lebih ketat
significant_lags = [lag for lag in range(1, len(acf_values)) if abs(acf_values[lag]) >= threshold]

if significant_lags:
    optimal_lag = significant_lags[0]  # Ambil lag signifikan pertama
else:
    optimal_lag = 1  # Default jika tidak ada yang signifikan

print("Nilai optimal lag (orde model):", optimal_lag)

In [ ]:
# Membentuk FLR dengan lag dari ACF
def create_flr(data, lag):
    flrs = []
    for i in range(len(data) - lag):
        # Akses data menggunakan iloc untuk indeks numerik pada baris
        antecedent = tuple(data.iloc[i:i+lag, data.columns.get_loc('Fuzzified')].tolist())
        consequent = data.iloc[i+lag, data.columns.get_loc('Fuzzified')]
        flrs.append((antecedent, consequent))
    return flrs

# Gunakan kolom 'Fuzzified' untuk FLR
flr_result = create_flr(fuzzified_data, optimal_lag)

# Tampilkan FLR
df_flr = pd.DataFrame(flr_result, columns=["Antecedent", "Consequent"])
df_flr

In [ ]:
# Visualisasi ACF
plt.figure(figsize=(8, 4))
# Remove 'use_line_collection=True' for older Matplotlib versions
plt.stem(range(len(acf_values)), acf_values)
plt.axhline(y=0, color='black', linestyle='--')
plt.axhline(y=0.5, color='red', linestyle='--', label='Threshold +0.5')
plt.axhline(y=-0.5, color='red', linestyle='--', label='Threshold -0.5')
plt.title("Autocorrelation Function (ACF)")
plt.xlabel("Lag")
plt.ylabel("ACF Value")
plt.legend()
plt.grid(True)
plt.show()

> Build & Train LSTM

In [ ]:
# Siapkan Midpoints
fuzzy_intervals = {
    1: (7.81, 7.97), 2: (7.97, 8.13), 3: (8.13, 8.29), 4: (8.29, 8.45),
    5: (8.45, 8.61), 6: (8.61, 8.77), 7: (8.77, 9.09), 8: (9.09, 9.25),
    9: (9.25, 9.41), 10: (9.38, 9.55), 11: (9.41, 9.57), 12: (9.51, 9.73)
}
fuzzy_midpoints = {k: sum(v)/2 for k, v in fuzzy_intervals.items()}

# Ubah Consequent Menjadi Midpoint
df_flr["Midpoint"] = df_flr["Consequent"].map(fuzzy_midpoints)
df_flr["Log_Midpoint"] = np.log(df_flr["Midpoint"])

# Fungsi Sliding Window
def create_sliding_window(data, target, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(target[i + window_size])
    return np.array(X), np.array(y)

# Konversi FLR ke array
X_raw = np.array(df_flr["Antecedent"].tolist())
y_raw = np.array(df_flr["Consequent"].tolist())

# Normalisasi seluruh data sebelum split (agar tetap urut sebagai time series)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X_raw.reshape(-1, 1))
y_scaled = scaler_y.fit_transform(y_raw.reshape(-1, 1))

results = []

# Loop untuk mencari window size terbaik
for window_size in range(1, 8):
    # Buat sliding window
    X_seq, y_seq = create_sliding_window(X_scaled, y_scaled, window_size)

    # Cek apakah cukup data
    if len(X_seq) < 10:
        print(f"Window size {window_size} menghasilkan data terlalu sedikit. Lewati.")
        continue

    # Split data (train:val:test = 64%:16%:20%)
    train_size = int(len(X_seq) * 0.64)
    val_size = int(len(X_seq) * 0.16)

    X_train, y_train = X_seq[:train_size], y_seq[:train_size]
    X_val, y_val = X_seq[train_size:train_size + val_size], y_seq[train_size:train_size + val_size]
    X_test, y_test = X_seq[train_size + val_size:], y_seq[train_size + val_size:]

    # Model LSTM
    model = Sequential()
    model.add(LSTM(64, activation="relu", input_shape=(window_size, 1), return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(32, activation="relu"))
    model.add(Dropout(0.2))
    model.add(Dense(1))

    model.compile(optimizer=Adam(learning_rate=0.001), loss=Huber())

    # Training
    history = model.fit(X_train, y_train, epochs=100, batch_size=16, validation_data=(X_val, y_val), verbose=0)

    # Prediksi dan evaluasi
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results.append([window_size, rmse])
    print(f"Window size: {window_size}, RMSE: {rmse:.4f}")

# Tampilkan window terbaik
if results:
    best_window = min(results, key=lambda x: x[1])[0]
    print(f"\nWindow size terbaik berdasarkan RMSE: {best_window}")
else:
    print("\nTidak ada window size yang menghasilkan data cukup untuk training.")

# Plot hasil
if results:
    window_sizes = [r[0] for r in results]
    rmses = [r[1] for r in results]

    plt.figure(figsize=(8, 5))
    plt.plot(window_sizes, rmses, marker="o", color="blue")
    plt.title("Perbandingan RMSE terhadap Window Size")
    plt.xlabel("Window Size")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.xticks(window_sizes)
    plt.tight_layout()
    plt.show()

In [ ]:
# Save Model
best_rmse = float("inf")
best_model = None

for window_size in range(1, 8):
    ...
    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_model.save("bestlstmmodel.h5")

In [ ]:
model.summary()

# Visualisasi Model Training & Loss
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

> Mendapatkan Hasil Prediksi

In [ ]:
# Get predictions (still in scaled form)
y_pred_scaled = model.predict(X_test_seq)

# Tentukan panjang minimal antara y_test_seq dan y_pred_scaled
min_len = min(len(y_test_seq), len(y_pred_scaled))

# Slice keduanya agar panjangnya sama
y_actual_scaled = y_test_seq[:min_len].flatten()
y_pred_scaled = y_pred_scaled[:min_len].flatten()

# Buat DataFrame hasil prediksi dalam skala normalisasi (scaled)
df_predictions = pd.DataFrame({
    "Actual": y_actual_scaled,
    "Predicted": y_pred_scaled
})

df_predictions

> Denormalisasi

In [ ]:
# Denormalisasi hasil prediksi
# Reshape y_pred to be 2-dimensional before inverse transformation
y_pred_inverse = scaler_y.inverse_transform(y_pred.reshape(-1, 1))
y_actual_inverse = scaler_y.inverse_transform(y_test_seq)

# Buat DataFrame baru menggunakan nilai yang telah di-denormalisasi
df_denormalized = pd.DataFrame({
    # Use the inverse transformed values
    "Actual": y_actual_inverse.flatten(),
    "Predicted": y_pred_inverse.flatten()
})
df_denormalized

> Defuzzifikasi

In [ ]:
num_intervals = 12
min_val = y_pred_inverse.min()
max_val = y_pred_inverse.max()
step = (max_val - min_val) / num_intervals

fuzzy_intervals = {}
for i in range(num_intervals):
    fuzzy_intervals[i + 1] = (min_val + i*step, min_val + (i+1)*step)

# Buat map label ke nilai tengah interval
fuzzy_map = {k: (start + end) / 2 for k, (start, end) in fuzzy_intervals.items()}

# Fungsi untuk dapatkan label fuzzy berdasar interval menaungi dulu, jika tidak ada maka nearest
def get_label(value, fuzzy_intervals):
    for label, (start, end) in fuzzy_intervals.items():
        if start <= value <= end:
            return label
    # fallback cari nearest jika nilai di luar interval
    min_dist = float('inf')
    best_label = None
    for label, (start, end) in fuzzy_intervals.items():
        mid = (start + end) / 2
        dist = abs(value - mid)
        if dist < min_dist:
            min_dist = dist
            best_label = label
    return best_label

# Ambil label fuzzy untuk setiap prediksi log
labels = np.array([get_label(v, fuzzy_intervals) for v in y_pred_inverse.flatten()])

# Defuzzifikasi: map label ke nilai tengah interval (log scale)
log_pred = np.array([fuzzy_map[label] for label in labels])

# Konversi dari log ke rupiah
rupiah_pred = np.exp(log_pred)

# Baca data historis untuk scaling
data = pd.read_csv("/content/drive/MyDrive/indofood.csv")
historical_min = data["Close"].min()
historical_max = data["Close"].max()

# Scaling ke rentang harga historis
pred_min = rupiah_pred.min()
pred_max = rupiah_pred.max()

if pred_max == pred_min:
    print("Warning: pred_max == pred_min, hasil akan statik.")
    scaled_rupiah_pred = np.full_like(rupiah_pred, historical_min)
else:
    scaled_rupiah_pred = (rupiah_pred - pred_min) / (pred_max - pred_min) * (historical_max - historical_min) + historical_min

# Buat DataFrame hasil akhir
df_defuzzified = pd.DataFrame({
    "Actual": y_actual_inverse.flatten(),
    "Predicted": y_pred_inverse.flatten(),
    "Fuzzy Label": labels,
    "Defuzzified Log": log_pred,
    "Predicted Price (Rupiah)": scaled_rupiah_pred
})

print(df_defuzzified)

In [ ]:
# Save to csv
df_defuzzified["Predicted Price (Rupiah)"].to_csv("prediksirupiah.csv", index=False)

> Evaluasi MAPE

In [ ]:
# Evaluasi Model menggunakan MAPE
mape = mean_absolute_percentage_error(df_defuzzified["Actual"], df_defuzzified["Predicted"])
print(f"Mean Absolute Percentage Error (MAPE): {mape}%")

In [ ]:
# Evaluasi MSE
mse = mean_squared_error(df_defuzzified["Actual"], df_defuzzified["Predicted"])
print(f"Mean Squared Error (MSE): {mse}")

# Evaluasi MAE
mae = mean_absolute_error(df_defuzzified["Actual"], df_defuzzified["Predicted"])
print(f"Mean Absolute Error (MAE): {mae}")

# Evaluasi RMSE
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error (RMSE): {rmse}")

# Evaluasi MAPE
mape = mean_absolute_percentage_error(df_defuzzified["Actual"], df_defuzzified["Predicted"])
print(f"Mean Absolute Percentage Error (MAPE): {mape}%")

> Forecasting 7 Hari Kedepan

In [ ]:
future_steps = 7
timesteps = X_test_seq.shape[1]  # Jumlah timestep yang digunakan saat pelatihan
num_features = X_test_seq.shape[2]  # Jumlah fitur

# Ambil sequence terakhir dari data test
last_sequence = X_test_seq[-1]  # Bentuk: (timesteps, num_features)
future_predictions = []

current_sequence = last_sequence.copy()

# --- Prediksi 7 hari ke depan secara sekuensial --- #
for _ in range(future_steps):
    input_seq = np.expand_dims(current_sequence, axis=0)  # (1, timesteps, num_features)
    next_pred = model.predict(input_seq, verbose=0)

    future_predictions.append(next_pred[0, 0])  # Simpan nilai prediksi (skalar)

    # Update sequence: geser dan tambahkan prediksi
    next_input = np.append(current_sequence[1:], [[next_pred[0, 0]] + list(current_sequence[-1, 1:])], axis=0)
    current_sequence = next_input

# --- Denormalisasi hasil prediksi (log scale) --- #
future_log_pred = scaler_y.inverse_transform(np.array(future_predictions).reshape(-1, 1)).flatten()

# --- Konversi log ke nilai rupiah --- #
future_rupiah_pred = np.exp(future_log_pred)

# --- Scaling ke rentang historis untuk konsistensi tampilan --- #
# Baca data historis dari CSV
data = pd.read_csv("/content/drive/MyDrive/indofood.csv")  # Sesuaikan path
historical_min = data["Close"].min()
historical_max = data["Close"].max()

pred_min = future_rupiah_pred.min()
pred_max = future_rupiah_pred.max()

if pred_max == pred_min:
    print("Warning: pred_max == pred_min, hasil akan statik.")
    scaled_future_rupiah = np.full_like(future_rupiah_pred, historical_min)
else:
    scaled_future_rupiah = (future_rupiah_pred - pred_min) / (pred_max - pred_min) * (historical_max - historical_min) + historical_min

# --- Buat DataFrame hasil prediksi 7 hari ke depan --- #
last_date = pd.to_datetime(data["Date"].iloc[-1])
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=future_steps)

df_future = pd.DataFrame({
    "Date": future_dates,
    "Predicted Price (Rupiah)": scaled_future_rupiah
})

# Tampilkan hasil prediksi
print(df_future)

In [ ]:
# Save to csv
df_future.to_csv("7days.csv", index=False)

**RISK ANALYSIS - DATA AKTUAL**
____________________________________________________________________________

In [ ]:
# Tampilkan data
aktual = pd.read_csv("/content/drive/MyDrive/indofood.csv")
aktual

> Hitung Nilai Return Saham

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis, norm, kstest

aktual["LogReturn"] = np.log(aktual["Close"] / aktual["Close"].shift(1))
aktual.dropna(inplace=True)
print(aktual[["Close", "LogReturn"]])

> Hitung Mean dan Variance

In [ ]:
# Estimasi parameter: mean (μ) dan std dev (σ)
mu = aktual["LogReturn"].mean()
sigma = aktual["LogReturn"].std()
print(f"Mean return: {mu:.6f}")
print(f"Std deviation: {sigma:.6f}")

> Uji Normalitas : Kolmogorov Smirnov

In [ ]:
# Uji normalitas Kolmogorov-Smirnov (K-S test)
ks_stat, ks_pvalue = kstest(aktual["LogReturn"], 'norm', args=(mu, sigma))
print(f"\nKolmogorov-Smirnov Test: p-value = {ks_pvalue:.6f}")

# Interpretasi
alpha = 0.05
if ks_pvalue > alpha:
    print("Data berdistribusi normal (gagal tolak H0)")
else:
    print("Data tidak berdistribusi normal (tolak H0)")

> Jika Data Tidak Berdistribusi Normal Gunakan Z Alpha'

In [ ]:
# Tentukan z_alpha atau z_alpha' (Cornish-Fisher)
alpha = 0.05  # 95% confidence level
z_alpha = norm.ppf(alpha)  # Kuantil bawah (karena VaR melihat kerugian)

if ks_pvalue <= 0.05:
    print("Distribusi tidak normal, gunakan Cornish-Fisher adjustment.")
    skewness = skew(aktual["LogReturn"])
    excess_kurt = kurtosis(aktual["LogReturn"], fisher=True)
    z_alpha_cf = (
        z_alpha +
        (1/6)*(z_alpha**2 - 1)*skewness +
        (1/24)*(z_alpha**3 - 3*z_alpha)*excess_kurt -
        (1/36)*(2*z_alpha**3 - 5*z_alpha)*(skewness**2)
    )
else:
    print("Distribusi normal, gunakan z_alpha biasa.")
    z_alpha_cf = z_alpha

print(f"z_alpha: {z_alpha:.4f}")
print(f"z_alpha' (Cornish-Fisher): {z_alpha_cf:.4f}")

> Simulasi Return Sebanyak m

In [ ]:
# Parameter simulasi Monte Carlo
t = 5              # waktu (5 hari)
W0 = 1_000_000     # Nilai awal portofolio
m = 10000          # Jumlah simulasi

# Simulasi return (distribusi normal)
simulated_returns = np.random.normal(mu * t, sigma * np.sqrt(t), size=m)
print(f"Simulasi Return Sebanyak {m} hari:")

df_simulated_returns = pd.DataFrame({"Simulasi Return": simulated_returns})
df_simulated_returns

> Hitung Kerugian Maksimum dalam t Hari

In [ ]:
# Hitung R* dari simulasi (dengan asumsi Cornish-Fisher)
R_star_list = []
for _ in range(100):  # langkah 7: rata-rata dari banyak simulasi
    sim_returns = np.random.normal(mu * t, sigma * np.sqrt(t), m)
    R_star = np.quantile(sim_returns, alpha)
    R_star_list.append(R_star)

df_R_star = pd.DataFrame({"R_star": R_star_list})
df_R_star

> Hitung Rata Rata R*

In [ ]:
R_star_avg_aktual = np.mean(R_star_list)
print(f"Rata-rata R* (kerugian maksimum kuantil ke-alpha): {R_star_avg:.6f}")

> Hitung VaR Berdasarkan R* dan W0

In [ ]:
# Hitung VaR menggunakan z_alpha' (Cornish-Fisher)
VaR_cf_aktual = W0 * abs(z_alpha_cf * sigma * np.sqrt(t))
print(f"\nEstimated VaR (95%) using Cornish-Fisher: Rp {VaR_cf:,.2f}")

> Visualisasi

In [ ]:
plt.figure(figsize=(12, 6))

# Histogram dari simulated returns
plt.subplot(1, 2, 1)
plt.hist(simulated_returns, bins=50, edgecolor="black")
plt.axvline(R_star, color="red", linestyle="--", label=f"Empirical R* = {R_star:.4f}")
plt.title("Histogram of Simulated Returns")
plt.xlabel("Simulated Returns")
plt.ylabel("Frequency")
plt.legend()

# Plot distribusi kumulatif
plt.subplot(1, 2, 2)
sorted_returns = np.sort(simulated_returns)
cdf = np.arange(1, m + 1) / m
plt.plot(sorted_returns, cdf)
plt.axvline(R_star, color="red", linestyle="--", label=f"Empirical R* = {R_star:.4f}")
plt.axhline(alpha, color="gray", linestyle="--", label=f"α = {alpha}")
plt.title("Cumulative Distribution of Simulated Returns")
plt.xlabel("Simulated Returns")
plt.ylabel("Cumulative Probability")
plt.legend()

plt.tight_layout()
plt.show()

**RISK ANALYSIS : DATA PREDIKSI**
_______________________________________________________________________

In [ ]:
# Tampilkan data
predicted = pd.read_csv("prediksirupiah.csv")
predicted

> Hitung Nilai Return Saham

In [ ]:
# Hitung log return: R_t = (P_t - P_{t-1}) / P_{t-1}
predicted["LogReturn"] = predicted.pct_change()
predicted.dropna(inplace=True)

# Tampilkan hasil
print("Log Return Saham:")
print(predicted)

> Hitung Rata - Rata dan Varians

In [ ]:
# Estimasi parameter: mean (μ) dan std dev (σ)
mu = predicted["LogReturn"].mean()
sigma = predicted["LogReturn"].std()
print(f"Mean return: {mu:.6f}")
print(f"Std deviation: {sigma:.6f}")

> Uji Normalitas : Kolmogorov Smirnov

In [ ]:
# Uji normalitas Kolmogorov-Smirnov (K-S test)
ks_stat, ks_pvalue = kstest(predicted["LogReturn"], 'norm', args=(mu, sigma))
print(f"\nKolmogorov-Smirnov Test: p-value = {ks_pvalue:.6f}")

# Interpretasi
alpha = 0.05
if ks_pvalue > alpha:
    print("Data berdistribusi normal (gagal tolak H0)")
else:
    print("Data tidak berdistribusi normal (tolak H0)")

> Jika Data Tidak Berdistribusi Normal, Gunakan Z Alpha

In [ ]:
# Tentukan z_alpha atau z_alpha' (Cornish-Fisher)
alpha = 0.05  # 95% confidence level
z_alpha = norm.ppf(alpha)  # Kuantil bawah (karena VaR melihat kerugian)

if ks_pvalue <= 0.05:
    print("Distribusi tidak normal, gunakan Cornish-Fisher adjustment.")
    skewness = skew(predicted["LogReturn"])
    excess_kurt = kurtosis(predicted["LogReturn"], fisher=True)
    z_alpha_cf = (
        z_alpha +
        (1/6)*(z_alpha**2 - 1)*skewness +
        (1/24)*(z_alpha**3 - 3*z_alpha)*excess_kurt -
        (1/36)*(2*z_alpha**3 - 5*z_alpha)*(skewness**2)
    )
else:
    print("Distribusi normal, gunakan z_alpha biasa.")
    z_alpha_cf = z_alpha

print(f"z_alpha: {z_alpha:.4f}")
print(f"z_alpha' (Cornish-Fisher): {z_alpha_cf:.4f}")

> Simulasi Return Sebanyak m

In [ ]:
# Parameter simulasi Monte Carlo
t = 5              # waktu (5 hari)
W0 = 1_000_000     # Nilai awal portofolio
m = 10000          # Jumlah simulasi

# Simulasi return (distribusi normal)
simulated_returns = np.random.normal(mu * t, sigma * np.sqrt(t), size=m)
print(f"Simulasi Return Sebanyak {m} hari:")

df_simulated_returns = pd.DataFrame({"Simulasi Return": simulated_returns})
df_simulated_returns

> Hitung Kerugian Maksimum dalam t Hari

In [ ]:
# Hitung R* dari simulasi (dengan asumsi Cornish-Fisher)
R_star_list = []
for _ in range(100):  # langkah 7: rata-rata dari banyak simulasi
    sim_returns = np.random.normal(mu * t, sigma * np.sqrt(t), m)
    R_star = np.quantile(sim_returns, alpha)
    R_star_list.append(R_star)

df_R_star = pd.DataFrame({"R_star": R_star_list})
df_R_star

> Hitung Rata - Rata R*

In [ ]:
R_star_avg = np.mean(R_star_list)
print(f"Rata-rata R* (kerugian maksimum kuantil ke-alpha): {R_star_avg:.6f}")

> Hitung VaR Berdasarkan R* dan W0

In [ ]:
# Hitung VaR menggunakan z_alpha' (Cornish-Fisher)
VaR_cf = W0 * abs(z_alpha_cf * sigma * np.sqrt(t))
print(f"\nEstimated VaR (95%) using Cornish-Fisher: Rp {VaR_cf:,.2f}")

> Visualisasi

In [ ]:
plt.figure(figsize=(12, 6))

# Histogram dari simulated returns
plt.subplot(1, 2, 1)
plt.hist(simulated_returns, bins=50, edgecolor="black")
plt.axvline(R_star, color="red", linestyle="--", label=f"Empirical R* = {R_star:.4f}")
plt.title("Histogram of Simulated Returns")
plt.xlabel("Simulated Returns")
plt.ylabel("Frequency")
plt.legend()

# Plot distribusi kumulatif
plt.subplot(1, 2, 2)
sorted_returns = np.sort(simulated_returns)
cdf = np.arange(1, m + 1) / m
plt.plot(sorted_returns, cdf)
plt.axvline(R_star, color="red", linestyle="--", label=f"Empirical R* = {R_star:.4f}")
plt.axhline(alpha, color="gray", linestyle="--", label=f"α = {alpha}")
plt.title("Cumulative Distribution of Simulated Returns")
plt.xlabel("Simulated Returns")
plt.ylabel("Cumulative Probability")
plt.legend()

plt.tight_layout()
plt.show()

**BACKTESTING KUPIEC TEST**
_______________________________________________________________________________

In [ ]:
# Ambil data return log riil 100 hari terakhir
n = 100
aktual["LogReturn"] = np.log(aktual["Close"] / aktual["Close"].shift(1))
aktual.dropna(inplace=True)
actual_returns = aktual["LogReturn"].iloc[-n:].values

In [ ]:
# Parameter
alpha = 0.05       # Tingkat kepercayaan 95%
n = len(actual_returns)

# Hitung VaR harian dari distribusi empiris
VaR_series = np.percentile(actual_returns, alpha * 100)  # quantile ke-5 (karena 5% VaR)

# Hitung pelanggaran: ketika return aktual < VaR (kerugian lebih besar dari yang diharapkan)
violations = actual_returns < VaR_series
x_alpha = violations.sum()
y_alpha = x_alpha / n

# Uji Kupiec (Likelihood Ratio Unconditional Coverage Test)
numerator = (y_alpha ** x_alpha) * ((1 - y_alpha) ** (n - x_alpha))
denominator = (alpha ** x_alpha) * ((1 - alpha) ** (n - x_alpha))
LR_uc = -2 * np.log(numerator / denominator)

# Hitung p-value (uji chi-square dengan 1 derajat kebebasan)
p_value = 1 - chi2.cdf(LR_uc, df=1)

# Output hasil
print(f"VaR harian pada tingkat kepercayaan {int((1-alpha)*100)}%: {VaR_series:.4f}")
print(f"Jumlah pelanggaran: {x_alpha}")
print(f"Proporsi pelanggaran aktual: {y_alpha:.4f}")
print(f"LR_uc: {LR_uc:.4f}")
print(f"P-value: {p_value:.4f}")

# Interpretasi
if p_value < 0.05:
    print("Model VaR DITOLAK oleh Uji Kupiec (tidak valid).")
else:
    print("Model VaR DITERIMA oleh Uji Kupiec (valid).")

In [ ]:
# Gunakan 100 hari terakhir sebagai backtesting
n = 100
prediksirupiah = pd.read_csv("prediksirupiah.csv")
prediksirupiah["LogReturn"] = np.log(prediksirupiah["Predicted Price (Rupiah)"] / prediksirupiah["Predicted Price (Rupiah)"].shift(1))
prediksirupiah.dropna(inplace=True)

In [ ]:
# Parameter# Parameter
alpha = 0.05       # Tingkat kepercayaan 95%
n = len(prediksirupiah)

# Hitung VaR harian dari distribusi empiris
VaR_series = np.percentile(prediksirupiah, alpha * 100)  # quantile ke-5 (karena 5% VaR)

# Hitung pelanggaran: ketika return aktual < VaR (kerugian lebih besar dari yang diharapkan)
violations = actual_returns < VaR_series
x_alpha = violations.sum()
y_alpha = x_alpha / n

# Uji Kupiec (Likelihood Ratio Unconditional Coverage Test)
numerator = (y_alpha ** x_alpha) * ((1 - y_alpha) ** (n - x_alpha))
denominator = (alpha ** x_alpha) * ((1 - alpha) ** (n - x_alpha))
LR_uc = -2 * np.log(numerator / denominator)

# Hitung p-value (uji chi-square dengan 1 derajat kebebasan)
p_value = 1 - chi2.cdf(LR_uc, df=1)

# Output hasil
print(f"VaR harian pada tingkat kepercayaan {int((1-alpha)*100)}%: {VaR_series:.4f}")
print(f"Jumlah pelanggaran: {x_alpha}")
print(f"Proporsi pelanggaran aktual: {y_alpha:.4f}")
print(f"LR_uc: {LR_uc:.4f}")
print(f"P-value: {p_value:.4f}")

# Interpretasi
if p_value < 0.05:
    print("Model VaR DITOLAK oleh Uji Kupiec (tidak valid).")
else:
    print("Model VaR DITERIMA oleh Uji Kupiec (valid).")
